In [2]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git
%cd unstructuredata_tfidf_repo
!ls

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 54 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 25.71 KiB | 3.67 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/unstructuredata_tfidf_repo/unstructuredata_tfidf_repo
data	 notebooks  requirements.txt  task_to_do.txt
LICENSE  README.md  src		      tests


In [3]:
!pip install -q -r requirements.txt

In [5]:
import nltk
from nltk.tokenize import word_tokenize
from src.load_data import load_documents

nltk.download('punkt')
nltk.download('punkt_tab')

titles, documents = load_documents()

print("Total documents:", len(documents))

for title, doc in zip(titles, documents):
    tokens = word_tokenize(doc)
    print(title, ":", len(tokens), "tokens")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total documents: 10
AI in Healthcare : 101 tokens
Climate Change : 97 tokens
Basketball Strategy : 102 tokens
Italian Pasta : 108 tokens
Space Exploration : 101 tokens
Cybersecurity : 109 tokens
Music Theory : 111 tokens
Gardening Vegetables : 108 tokens
Economics Inflation : 109 tokens
Public Health : 115 tokens


In [6]:
print("Total documents:", len(documents))
print("Total titles:", len(titles))

for i, (title, doc) in enumerate(zip(titles, documents), 1):
    print(i, title, ":", len(doc.split()), "words")

assert len(documents) == 10
assert len(titles) == 10
assert all(len(doc.split()) > 30 for doc in documents)

print("\nVerification Passed: All 10 documents loaded!")

Total documents: 10
Total titles: 10
1 AI in Healthcare : 85 words
2 Climate Change : 83 words
3 Basketball Strategy : 85 words
4 Italian Pasta : 91 words
5 Space Exploration : 86 words
6 Cybersecurity : 88 words
7 Music Theory : 87 words
8 Gardening Vegetables : 89 words
9 Economics Inflation : 89 words
10 Public Health : 91 words

Verification Passed: All 10 documents loaded!


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(documents)

print("Matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Matrix shape: (10, 537)
Vocabulary size: 537


In [8]:
import pandas as pd

df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=titles,
    columns=vectorizer.get_feature_names_out()
)

print(df.iloc[:3, :10])

                      ability  accelerate  access  accountability  accuracy  \
AI in Healthcare     0.000000         0.0     0.0          0.1246    0.1246   
Climate Change       0.000000         0.0     0.0          0.0000    0.0000   
Basketball Strategy  0.130137         0.0     0.0          0.0000    0.0000   

                     accurate   actions  activity  adaptation  add  
AI in Healthcare     0.000000  0.000000       0.0    0.000000  0.0  
Climate Change       0.124795  0.124795       0.0    0.124795  0.0  
Basketball Strategy  0.000000  0.000000       0.0    0.000000  0.0  


In [9]:
for title in titles:
    print(title, ":", list(df.loc[title].nlargest(5).index))

AI in Healthcare : ['ai', 'healthcare', 'use', 'accountability', 'accuracy']
Climate Change : ['change', 'emissions', 'climate', 'energy', 'accurate']
Basketball Strategy : ['ability', 'analytics', 'basketball', 'breaks', 'chemistry']
Italian Pasta : ['pasta', 'sauces', 'sauce', 'different', 'add']
Space Exploration : ['space', 'mars', 'missions', 'accelerate', 'advanced']
Cybersecurity : ['cybersecurity', 'systems', 'data', 'attackers', 'attacks']
Music Theory : ['music', 'musicians', 'practice', 'allows', 'artists']
Gardening Vegetables : ['soil', 'pests', 'year', 'improves', 'adds']
Economics Inflation : ['inflation', 'spending', 'economic', 'trade', 'affect']
Public Health : ['health', 'public', 'sleep', 'access', 'activity']


In [10]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
from nltk.tokenize import word_tokenize

all_tokens = []

for doc in documents:
    tokens = word_tokenize(doc.lower())
    all_tokens.extend(tokens)

# Find stop words and count their occurrences
stopword_counts = Counter(
    word for word in all_tokens
    if word in ENGLISH_STOP_WORDS
)

print("Total stop words removed:", sum(stopword_counts.values()))
print("Unique stop words removed:", len(stopword_counts))

print("\nStop words and their frequencies:")
for word, count in stopword_counts.most_common():
    print(word, ":", count)

Total stop words removed: 222
Unique stop words removed: 61

Stop words and their frequencies:
and : 76
to : 12
can : 8
on : 8
a : 8
are : 6
for : 6
the : 6
is : 5
of : 5
by : 4
from : 4
as : 4
in : 4
with : 4
that : 3
more : 3
also : 3
or : 3
rather : 2
than : 2
between : 2
often : 2
be : 2
through : 2
at : 2
over : 2
find : 1
might : 1
however : 1
before : 1
if : 1
them : 1
such : 1
but : 1
toward : 1
while : 1
now : 1
about : 1
sometimes : 1
should : 1
until : 1
then : 1
across : 1
only : 1
has : 1
may : 1
so : 1
become : 1
must : 1
becomes : 1
both : 1
well : 1
many : 1
each : 1
after : 1
how : 1
interest : 1
always : 1
system : 1
whole : 1


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def search_documents(query):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix)[0]

    results = sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)

    return results

queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet"
]

for query in queries:
    print("\nQuery:", query)
    for title, score in search_documents(query):
        print(title, ":", round(score, 4))


Query: renewable energy and climate policy
Climate Change : 0.3235
Economics Inflation : 0.0952
Gardening Vegetables : 0.0485
AI in Healthcare : 0.0
Basketball Strategy : 0.0
Italian Pasta : 0.0
Space Exploration : 0.0
Cybersecurity : 0.0
Music Theory : 0.0
Public Health : 0.0

Query: password security and phishing
Cybersecurity : 0.1764
AI in Healthcare : 0.0
Climate Change : 0.0
Basketball Strategy : 0.0
Italian Pasta : 0.0
Space Exploration : 0.0
Music Theory : 0.0
Gardening Vegetables : 0.0
Economics Inflation : 0.0
Public Health : 0.0

Query: healthy sleep exercise and diet
Public Health : 0.2849
Gardening Vegetables : 0.0447
AI in Healthcare : 0.0
Climate Change : 0.0
Basketball Strategy : 0.0
Italian Pasta : 0.0
Space Exploration : 0.0
Cybersecurity : 0.0
Music Theory : 0.0
Economics Inflation : 0.0


In [14]:
assert search_documents("renewable energy and climate policy")[0][0] == "Climate Change"
assert search_documents("password security and phishing")[0][0] == "Cybersecurity"
assert search_documents("healthy sleep exercise and diet")[0][0] == "Public Health"

print("All three queries verified successfully!")

All three queries verified successfully!


In [12]:
new_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

new_matrix = new_vectorizer.fit_transform(documents)

print("Original vocabulary:", len(vectorizer.get_feature_names_out()))
print("New vocabulary:", len(new_vectorizer.get_feature_names_out()))

new_df = pd.DataFrame(
    new_matrix.toarray(),
    index=titles,
    columns=new_vectorizer.get_feature_names_out()
)

for title in titles:
    print(title, ":", list(new_df.loc[title].nlargest(5).index))

Original vocabulary: 537
New vocabulary: 69
AI in Healthcare : ['use', 'careful', 'clean', 'disease', 'improve']
Climate Change : ['climate', 'energy', 'economic', 'economic growth', 'efficiency']
Basketball Strategy : ['adjust', 'based', 'communication', 'decisions', 'defenses']
Italian Pasta : ['different', 'flavor', 'improve', 'long', 'quality']
Space Exploration : ['international', 'medical', 'health', 'data', 'water']
Cybersecurity : ['systems', 'data', 'defenses', 'essential', 'intelligence']
Music Theory : ['practice', 'communication', 'different', 'improves', 'live']
Gardening Vegetables : ['improves', 'careful', 'climate', 'flavor', 'healthy']
Economics Inflation : ['economic', 'trade', 'adjust', 'decisions', 'economic growth']
Public Health : ['health', 'based', 'clean', 'disease', 'healthy']


In [15]:
!pytest -q

...                                                                      [100%]
3 passed in 1.41s
